# 02 — Audio Signal Exploration

**Aim.** Understand what acoustic measurements capture in four **controlled** RAVDESS recordings (same actor, statement, repetition and intensity; differing emotion only). These plots are illustrative, **not** population-level evidence.

From *Implementing the Amplitude Envelope*, *RMS Energy and Zero-Crossing Rate*, *Working with Audio in Python* and *Intro to Audio Processing*, we adapt waveform overlays, frame-level measurements, and the STFT → Mel explanation to **speech**, not the original music examples.

## 1. Paths and matched examples

In [ ]:
from pathlib import Path
import zipfile

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import drive
from IPython.display import Audio, display

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/speech-emotion-recognition-project-d')
ZIP_PATH = PROJECT_DIR / 'data/Audio_Speech_Actors_01-24.zip'
METADATA_PATH = PROJECT_DIR / 'data/processed/ravdess_metadata.csv'
EXTRACT_DIR = Path('/content/ravdess_speech')
FIGURES_DIR = PROJECT_DIR / 'results/figures/02_audio'

assert METADATA_PATH.is_file(), 'Run notebook 01 first.'
metadata = pd.read_csv(METADATA_PATH)
if len(list(EXTRACT_DIR.rglob('*.wav'))) != 1440:
    assert ZIP_PATH.is_file(), ZIP_PATH
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(EXTRACT_DIR)
assert len(list(EXTRACT_DIR.rglob('*.wav'))) == 1440
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR = 16000  # Same sample rate as feature extraction in 03
N_FFT = 1024
HOP_LENGTH = 256
FRAME_LENGTH = 1024
EMOTIONS = ['neutral', 'happy', 'sad', 'angry']

samples = {}
for emotion in EMOTIONS:
    row = metadata.loc[
        (metadata.emotion == emotion)
        & (metadata.actor_id == 1)
        & (metadata.statement == 1)
        & (metadata.repetition == 1)
        & (metadata.intensity == 1)
    ]
    assert len(row) == 1, emotion
    path = EXTRACT_DIR / row.iloc[0]['relative_path']
    samples[emotion], _ = librosa.load(path, sr=TARGET_SR, mono=True)
    print(f'{emotion}: {len(samples[emotion])/TARGET_SR:.2f} s')
    display(Audio(samples[emotion], rate=TARGET_SR, normalize=False))

## 2. Waveforms and short-time statistics

Waveform amplitude is not the same as loudness. At a fixed window length $N$ and hop $H$:

- **Amplitude envelope**: $AE_m=\max_{0\le n<N}|x[mH+n]|$ — peak magnitude in a frame.
- **RMS**: $RMS_m=\sqrt{\frac{1}{N}\sum_{n=0}^{N-1}x[mH+n]^2}$ — local signal level.
- **ZCR**: normalized number of sign changes inside a frame; affected by voicing, noise and signal conditions, **not** a direct emotion detector.

We use the **absolute** peak, not `max(frame)` (which would ignore large negative peaks). All three use the same frame size and hop, and `center=False` aligns their timestamps. The red AE/RMS curves can be overlaid on waveforms; ZCR has a separate axis because it has different units.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 11), constrained_layout=True)

for i, emotion in enumerate(EMOTIONS):
    y = samples[emotion]
    starts = np.arange(0, len(y) - FRAME_LENGTH + 1, HOP_LENGTH)
    times = (starts + FRAME_LENGTH / 2) / TARGET_SR
    envelope = np.array([np.max(np.abs(y[s:s + FRAME_LENGTH])) for s in starts])
    rms = librosa.feature.rms(y=y, frame_length=FRAME_LENGTH,
                               hop_length=HOP_LENGTH, center=False)[0]
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=FRAME_LENGTH,
                                             hop_length=HOP_LENGTH, center=False)[0]
    amplitude_limit = max(np.max(np.abs(y)), np.max(envelope)) * 1.15

    for j, values in enumerate((envelope, rms)):
        ax = axes[i, j]
        librosa.display.waveshow(y, sr=TARGET_SR, ax=ax, color='#A6BCCD', alpha=.7)
        ax.plot(times, values, color='#B57EDC', lw=1.6)
        ax.set_ylim(-amplitude_limit, amplitude_limit)
        ax.set_ylabel('Amplitude')

    ax = axes[i, 2]
    ax.plot(times, zcr, color='#9461B5', lw=1.4)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Zero-crossing rate')

    for ax, label in zip(axes[i], ('Amplitude envelope', 'RMS', 'ZCR')):
        ax.set_title(f'{emotion.capitalize()} — {label}', fontweight='bold', loc='left')
        ax.set_xlabel('Time (s)')
        ax.grid(axis='y', alpha=.15, linestyle='--')
        ax.spines[['top', 'right', 'left']].set_visible(False)
        ax.tick_params(length=0)

fig.savefig(FIGURES_DIR / 'waveform_amplitude_rms_zcr.png', dpi=170)
plt.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

labels = ("Amplitude envelope", "RMS", "ZCR")

fig = make_subplots(
    rows=len(EMOTIONS), cols=3,
    shared_xaxes="rows",
    subplot_titles=[
        f"{emotion.capitalize()} — {label}"
        for emotion in EMOTIONS for label in labels
    ],
    horizontal_spacing=0.07,
    vertical_spacing=0.07
)

for i, emotion in enumerate(EMOTIONS, start=1):
    y = samples[emotion]
    starts = np.arange(0, len(y) - FRAME_LENGTH + 1, HOP_LENGTH)
    times = (starts + FRAME_LENGTH / 2) / TARGET_SR

    envelope = np.array([
        np.max(np.abs(y[s:s + FRAME_LENGTH])) for s in starts
    ])
    rms = librosa.feature.rms(
        y=y, frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH, center=False
    )[0]
    zcr = librosa.feature.zero_crossing_rate(
        y, frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH, center=False
    )[0]

    # Проріджуємо лише фон для швидкого відображення
    step = max(1, int(np.ceil(len(y) / 15000)))
    waveform_times = np.arange(0, len(y), step) / TARGET_SR
    amplitude_limit = np.max(np.abs(y)) * 1.15

    for j, values in enumerate((envelope, rms, zcr), start=1):
        if j < 3:
            fig.add_trace(go.Scatter(
                x=waveform_times, y=y[::step],
                mode="lines", line=dict(color="#A6BCCD", width=1),
                opacity=0.45, hoverinfo="skip", showlegend=False
            ), row=i, col=j)

        fig.add_trace(go.Scatter(
            x=times, y=values, mode="lines", name=labels[j - 1],
            line=dict(color="#9461B5", width=2),
            hovertemplate="Time: %{x:.3f} s<br>Value: %{y:.4f}<extra></extra>",
            showlegend=False
        ), row=i, col=j)

        fig.update_yaxes(
            range=[0, 1] if j == 3 else [-amplitude_limit, amplitude_limit],
            title_text="ZCR" if j == 3 else "Amplitude",
            row=i, col=j
        )
        fig.update_xaxes(
            title_text="Time (s)", range=[0, len(y) / TARGET_SR],
            row=i, col=j
        )

fig.update_layout(
    template="simple_white",
    height=1150,
    margin=dict(l=60, r=25, t=60, b=45),
    font=dict(size=12),
    hovermode="x",
    dragmode="zoom"
)
fig.update_yaxes(showgrid=True, gridcolor="#EEEEEE", zeroline=False)

fig.show(
    renderer="colab",
    config={"scrollZoom": True, "displaylogo": False}
)

### A small check: RMS formula vs librosa

This checks the mathematics on exactly the same **non-centered** frames. It is a sanity check, not an additional feature-engineering experiment.

In [ ]:
y = samples['neutral']
starts = np.arange(0, len(y) - FRAME_LENGTH + 1, HOP_LENGTH)
rms_manual = np.array([
    np.sqrt(np.mean(y[s:s + FRAME_LENGTH] ** 2)) for s in starts
])
rms_librosa = librosa.feature.rms(
    y=y, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH, center=False
)[0]
print('Maximum absolute RMS difference:', np.max(np.abs(rms_manual - rms_librosa)))
assert np.allclose(rms_manual, rms_librosa, atol=1e-6)

## 3. Fourier → STFT → Mel

The discrete Fourier transform represents a windowed signal as frequency components:

$$X[k]=\sum_{n=0}^{N-1}x[n]e^{-j2\pi kn/N}.$$

Because speech is nonstationary, the **short-time Fourier transform (STFT)** computes a spectrum over overlapping windows. Its window length controls the time–frequency resolution trade-off. In `03` we also use `n_fft=1024`, `hop_length=256` at 16 kHz.

The **Mel spectrogram** applies filters on a nonlinear perceptual frequency scale to the **power spectrogram**. Therefore use `librosa.power_to_db` for Mel power and `librosa.amplitude_to_db` for STFT magnitude. These conversions are *not interchangeable*. Color limits below are shared across emotions within each spectrogram type.

Each spectrogram is converted to dB **relative to that recording’s own maximum**; therefore compare spectral patterns, not absolute sound levels, from these figures.

In [ ]:
stft_db, mel_db = {}, {}
for emotion, y in samples.items():
    magnitude = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH))
    power_mel = librosa.feature.melspectrogram(
        y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=64, fmax=TARGET_SR / 2
    )
    stft_db[emotion] = librosa.amplitude_to_db(magnitude, ref=np.max)
    mel_db[emotion] = librosa.power_to_db(power_mel, ref=np.max)

fig, axes = plt.subplots(4, 2, figsize=(14, 12), constrained_layout=True)
for i, emotion in enumerate(EMOTIONS):
    a = librosa.display.specshow(stft_db[emotion], sr=TARGET_SR,
        hop_length=HOP_LENGTH, x_axis='time', y_axis='log',
        vmin=-80, vmax=0, cmap='magma', ax=axes[i, 0])
    b = librosa.display.specshow(mel_db[emotion], sr=TARGET_SR,
        hop_length=HOP_LENGTH, x_axis='time', y_axis='mel',
        vmin=-80, vmax=0, cmap='magma', ax=axes[i, 1])
    axes[i, 0].set_title(f'{emotion.capitalize()} — STFT magnitude (dB)')
    axes[i, 1].set_title(f'{emotion.capitalize()} — Mel power (dB)')
fig.colorbar(
    a,
    ax=axes.ravel().tolist(),
    location="top",
    orientation="horizontal",
    label="Level relative to each panel's maximum (dB)",
    ticks=np.arange(-80, 1, 20),
    shrink=0.7,
    aspect=50,
    pad=0.04
)
fig.savefig(FIGURES_DIR / 'stft_vs_mel.png', dpi=170)
plt.show()

In [ ]:
stft_db, mel_db = {}, {}

for emotion in EMOTIONS:
    y = samples[emotion]

    magnitude = np.abs(librosa.stft(
        y, n_fft=N_FFT, hop_length=HOP_LENGTH
    ))
    power_mel = librosa.feature.melspectrogram(
        y=y, sr=TARGET_SR,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=64, fmax=TARGET_SR / 2
    )

    # Кожну панель нормалізуємо до її власного максимуму
    stft_db[emotion] = librosa.amplitude_to_db(magnitude, ref=np.max)
    mel_db[emotion] = librosa.power_to_db(power_mel, ref=np.max)

max_freq = TARGET_SR / 2

fig, axes = plt.subplots(
    len(EMOTIONS), 2, figsize=(14, 12),
    constrained_layout=True, squeeze=False
)

for i, emotion in enumerate(EMOTIONS):
    for j, (data, scale, title) in enumerate([
        (stft_db[emotion], "log", "STFT magnitude (dB)"),
        (mel_db[emotion], "mel", "Mel power (dB)")
    ]):
        ax = axes[i, j]

        image = librosa.display.specshow(
            data, sr=TARGET_SR, hop_length=HOP_LENGTH,
            x_axis="time", y_axis=scale,
            vmin=-80, vmax=0, cmap="magma", ax=ax
        )

        # Детальніші поділки на низьких частотах для log scale
        tick_values = (
            [0, 64, 128, 256, 512, 1024, 2048, 4096]
            if j == 0 else
            [0, 512, 1024, 2048, 4096]
        )
        ticks = [f for f in tick_values if f < max_freq] + [max_freq]

        ax.set_ylim(0, max_freq)
        ax.set_yticks(ticks, labels=[f"{f:g}" for f in ticks])
        ax.minorticks_off()
        ax.set_ylabel("Hz")
        ax.set_xlabel("Time (s)")
        ax.set_title(
            f"{emotion.capitalize()} — {title}", loc="center"
        )

        ax.text(
            0.98, 0.95,
            "log scale" if j == 0 else "Mel scale",
            transform=ax.transAxes,
            ha="right", va="top", color="white", fontsize=10,
            bbox=dict(
                facecolor="black", alpha=0.5, edgecolor="none"
            )
        )

fig.colorbar(
    image, ax=axes.ravel().tolist(),
    location="top", orientation="horizontal",
    label="Level relative to each panel's maximum (dB)",
    ticks=np.arange(-80, 1, 20),
    shrink=0.7, aspect=50, pad=0.04
)

fig.savefig(FIGURES_DIR / "stft_vs_mel.png", dpi=170)
plt.show()

## 4. What leading/trailing silence trimming changes

`librosa.effects.trim(..., top_db=30)` keeps the segment between detected non-silent edges; **it does not remove internal pauses**. These four examples illustrate the operation. The distribution for **all 1,440 recordings** belongs to `03`, alongside the actual preprocessing pipeline.

In [ ]:
trim_rows = []
fig, axes = plt.subplots(4, 2, figsize=(13, 10), constrained_layout=True)

for i, emotion in enumerate(EMOTIONS):
    y = samples[emotion]
    trimmed, _ = librosa.effects.trim(y, top_db=30)
    trim_rows.append({
        'emotion': emotion,
        'original_sec': len(y) / TARGET_SR,
        'trimmed_sec': len(trimmed) / TARGET_SR,
        'removed_pct': 100 * (1 - len(trimmed) / len(y))
    })
    limit = max(np.max(np.abs(y)) * 1.15, 1e-6)  # Same amplitude scale in both panels

    for ax, signal, label in zip(axes[i], (y, trimmed), ('Original', 'Trimmed')):
        librosa.display.waveshow(signal, sr=TARGET_SR, ax=ax, color='#B57EDC')
        ax.set_ylim(-limit, limit)
        ax.set_title(f'{emotion.capitalize()} — {label}', fontweight='bold', loc='left')
        ax.set_xlabel('Time from displayed signal start (s)')
        ax.grid(axis='y', alpha=.15, linestyle='--')
        ax.spines[['top', 'right', 'left']].set_visible(False)
        ax.tick_params(length=0)

fig.savefig(FIGURES_DIR / 'trimming_examples.png', dpi=170)
plt.show()
display(pd.DataFrame(trim_rows).round(2))


## 5. Fundamental frequency (F0) — voiced frames only

$F_0=1/T_0$ is the approximate fundamental frequency of periodic vocal-fold vibration. F0 may depend on emotion, actor, sentence and intensity; comparison of four recordings **cannot** establish a dataset-wide pattern. Unvoiced frames are shown as gaps.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharey=True)
pitch_rows = []
for ax, (emotion, y) in zip(axes.flat, samples.items()):
    f0, voiced, _ = librosa.pyin(y, sr=TARGET_SR, fmin=65, fmax=500,
                                frame_length=2048, hop_length=512)
    times = librosa.times_like(f0, sr=TARGET_SR, hop_length=512)
    valid = f0[np.isfinite(f0)]
    pitch_rows.append({'emotion': emotion,
        'f0_mean_hz': np.mean(valid) if valid.size else np.nan,
        'f0_std_hz': np.std(valid) if valid.size else np.nan,
        'voiced_ratio': np.mean(voiced)})
    ax.plot(times, f0, color='#a94d57', linewidth=1.2)
    ax.set_title(emotion.capitalize())
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('F0 (Hz)')
plt.suptitle('Estimated pitch in voiced frames (pYIN)')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'pitch_f0_examples.png', dpi=170)
plt.show()
display(pd.DataFrame(pitch_rows).round(2))

### What these figures do—and do not—show

AE and RMS track local changes in amplitude, ZCR tracks sign changes, spectrograms show time–frequency structure, and F0 estimates voiced pitch. **None is a standalone emotion label.** These four recordings are controlled demonstrations; in `03` we extract aggregated features and examine distributions over all actors, then in `04–05` test predictive value using actor-independent validation.

**Sources adapted:** Valerio Velardo, *Audio Signal Processing for Machine Learning* (amplitude envelope, RMS, ZCR); the provided *Working with Audio in Python* and *Intro to Audio Processing* notebooks (trim, STFT, Mel, Nyquist). The formulas and plotting code here are independently adapted to RAVDESS speech.

https://github.com/musikalkemist/AudioSignalProcessingForML